# 🐾 SISCO — Demo en Google Colab
## Chatbot de Detección de Estrés Académico — FUP

### ▶️ Instrucciones de uso:
1. Ejecuta las celdas **en orden** (clic en ▶ de cada celda, o `Ctrl+F9` para todas)
2. La **Celda 4** tardará ~2-3 minutos en cargar el modelo BETO
3. Al final aparecerá una **URL pública** tipo `https://xxxx.ngrok-free.app`
4. ¡Comparte esa URL con quien quieras!

> ⚠️ **Importante:** Mantén esta pestaña de Colab abierta. Si la cierras, el servidor se apaga.

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 1: Instalar dependencias
# ══════════════════════════════════════════════════════════
print('📦 Instalando dependencias... (puede tardar 2-3 minutos)')
import subprocess
subprocess.run(['pip', 'install', '-q',
    'flask==3.0.3',
    'transformers==4.46.0',
    'torch==2.3.0',
    'scikit-learn==1.5.2',
    'gdown==5.1.0',
    'pyngrok',
    'gunicorn'
], check=True)
print('✅ Dependencias instaladas correctamente')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 2: Descargar el proyecto desde GitHub
# ══════════════════════════════════════════════════════════
import os, subprocess

REPO_URL = 'https://github.com/mariapapamija5-dot/Chatbot-InventarioSISCO.git'
PROYECTO_DIR = '/content/sisco'

if os.path.exists(PROYECTO_DIR):
    print('🔄 Actualizando repositorio existente...')
    subprocess.run(['git', '-C', PROYECTO_DIR, 'pull'], check=True)
else:
    print('📥 Clonando repositorio desde GitHub...')
    subprocess.run(['git', 'clone', REPO_URL, PROYECTO_DIR], check=True)

os.chdir(PROYECTO_DIR)
print(f'✅ Proyecto listo en: {os.getcwd()}')
print('📂 Archivos:', [f for f in os.listdir('.') if not f.startswith('.')])

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 3: Descargar pesos del modelo desde Google Drive
# ══════════════════════════════════════════════════════════
import os, gdown

# ID del archivo pesos.pt en Google Drive
GOOGLE_DRIVE_ID = '1ot-_JNAaSGg_FS0eoqrLLLsM8FD8iBEM'
PESOS_PATH = '/content/sisco/modelo_hibrido_final/pesos.pt'

if os.path.exists(PESOS_PATH):
    size_mb = os.path.getsize(PESOS_PATH) / (1024*1024)
    print(f'✅ pesos.pt ya existe ({size_mb:.0f} MB) — omitiendo descarga')
else:
    print('📥 Descargando pesos del modelo (440 MB, ~2-4 min según conexión)...')
    url = f'https://drive.google.com/uc?id={GOOGLE_DRIVE_ID}'
    gdown.download(url, PESOS_PATH, quiet=False)
    size_mb = os.path.getsize(PESOS_PATH) / (1024*1024)
    print(f'✅ Pesos descargados: {size_mb:.0f} MB')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4: Configurar ngrok
# ══════════════════════════════════════════════════════════
# 🔑 INSTRUCCIONES PARA OBTENER TU TOKEN:
#   1. Ve a https://ngrok.com/ y crea una cuenta GRATIS
#   2. En el dashboard, clic en "Your Authtoken" en el menú izquierdo
#   3. Copia el token y pégalo abajo entre las comillas

NGROK_TOKEN = ''  # <-- PON AQUÍ TU TOKEN DE NGROK

# ─────────────────────────────────────────────────────────
if not NGROK_TOKEN.strip():
    raise ValueError(
        '❌ Falta el token de ngrok.\n'
        'Ve a https://ngrok.com → Sign up (gratis) → Dashboard → "Your Authtoken"\n'
        'Copia el token y pégalo en la variable NGROK_TOKEN de esta celda.'
    )

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('✅ Token de ngrok configurado')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 5: Iniciar SISCO con URL pública
# ══════════════════════════════════════════════════════════
import os, sys, threading, time, subprocess
from pyngrok import ngrok

os.chdir('/content/sisco')
sys.path.insert(0, '/content/sisco')

PUERTO = 7860

print('🐾 Iniciando servidor SISCO...')
print('   (Cargando PyTorch + modelo BETO, espera 2-3 minutos)\n')

# Iniciar Flask en un hilo de fondo
env = os.environ.copy()
env['FLASK_ENV'] = 'production'
env['PYTHONPATH'] = '/content/sisco'

flask_proc = subprocess.Popen(
    [sys.executable, 'app.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Esperar a que Flask esté listo leyendo su stdout en tiempo real
inicio = time.time()
listo = False
for linea in flask_proc.stdout:
    print(f'   [SISCO] {linea.rstrip()}')
    if 'Running on' in linea or 'Modelo hibrido cargado' in linea or 'SISCO' in linea:
        if 'cargado' in linea.lower() or 'running' in linea.lower():
            listo = True
    if listo and (time.time() - inicio) > 10:
        break
    if (time.time() - inicio) > 180:  # Timeout 3 min
        print('⚠️ Timeout esperando Flask, creando túnel de todas formas...')
        break

# Dar 5 segundos extra para asegurar que Flask está escuchando
time.sleep(5)

# Crear el túnel público con ngrok
try:
    tunel = ngrok.connect(PUERTO)
    url_publica = tunel.public_url
    if not url_publica.startswith('https'):
        url_publica = url_publica.replace('http://', 'https://')
    
    print('\n' + '='*60)
    print('🎉  ¡SISCO está disponible en la siguiente URL!')
    print(f'\n   👉  {url_publica}')
    print('\n' + '='*60)
    print('📋  Comparte este enlace con el jurado o estudiantes.')
    print('⚠️   Mantén esta celda corriendo. Si la detienes, el')
    print('    servidor se apaga y la URL deja de funcionar.')
    print('='*60)
except Exception as e:
    print(f'❌ Error creando túnel ngrok: {e}')
    print('   Verifica que tu token es correcto en la Celda 4.')

# Mantener el proceso Flask vivo
flask_proc.wait()

---
## ❓ Solución de problemas frecuentes

| Problema | Solución |
|:---|:---|
| `ValueError: Falta el token` | Pega tu token de ngrok en la Celda 4 |
| `gdown` falla descargando pesos | El archivo de Google Drive puede requerir acceso público. Verifica que el enlace sea compartido como "Cualquiera con el enlace" |
| Flask da error al iniciar | Verifica en los logs si dice "ModuleNotFoundError" y ejecuta de nuevo la Celda 1 |
| La URL ngrok da `502 Bad Gateway` | El modelo aún está cargando. Espera 1 minuto más y recarga la página |
| La sesión de Colab expiró | Vuelve a ejecutar todas las celdas desde el inicio (los pesos no se redescargan si ya existen) |
